<h1 style="color:black; font-family:Arial, sans-serif;">🏡 Scraping Immobilier – Avito.ma</h1>

<hr style="border:1px solid #ccc;">

<h2 style="color:#117A65;">Data Transformation</h2>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import re
from scipy import stats
import mysql.connector

In [ ]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="avito_db"
)

cursor = conn.cursor()

In [ ]:
query = "SELECT * FROM Avito_terrain"
df = pd.read_sql(query, conn)
df.head()

In [ ]:
df.shape[0]

In [ ]:
df.columns

In [ ]:
import warnings

warnings.filterwarnings('ignore')


In [ ]:
# Replace 'Nan' strings with actual NaN values
df = df.replace('Nan', np.nan)

In [ ]:
df = df[~df['Surface_Totale'].isna()]
print(df.shape[0])

In [ ]:
df['Ville'].unique()

In [ ]:
df['Ville'].value_counts()

In [ ]:
counts = df['Ville'].value_counts()
villes_a_remplacer = counts[counts <= 20].index

# Remplacer dans le DataFrame
df['Ville'] = df['Ville'].replace(villes_a_remplacer, 'Autres')

In [ ]:
df['Ville'].value_counts()

In [ ]:
mapping_villes = {
    "Bouskoura": "Casablanca élargi",
    "Bouznika": "Casablanca élargi",
    "Berrechid": "Casablanca élargi",
    "Settat": "Casablanca élargi",
    "Sidi Rahal": "Casablanca élargi",
    "Benslimane": "Casablanca élargi",
    "Bir Jdid": "Casablanca élargi",
    "Temara": "Rabat élargi",
    "Sidi Allal El Bahraoui": "Rabat élargi",
    "Tifelt": "Rabat élargi",
    "Salé": "Rabat élargi",
    "Rabat": "Rabat élargi",
    "Tétouan": "Tanger élargi",
    "Asilah": "Tanger élargi",
}

df['Ville'] = df['Ville'].replace(mapping_villes)

df['Ville'].value_counts()


In [ ]:
mapping_villes = {
    "Casablanca élargi": "Casablanca et alentours",
    "Casablanca": "Casablanca et alentours",
    "Rabat élargi": "Rabat et alentours",
    "Rabat": "Rabat et alentours",
    "Tanger élargi": "Tanger et alentours",
    "Tanger": "Tanger et alentours",
    "Fès" : "Fès-Meknès",
    "Meknès" : "Fès-Meknès",
    "El Jadida": "Autres",
    "Agadir": "Autres"
}

df['Ville'] = df['Ville'].replace(mapping_villes)

df['Ville'].value_counts()


In [ ]:
# Calculate the number of nulls in each column
nulls_per_column = df.isnull().sum()

# Calculate the total number of nulls in the DataFrame
total_nulls = nulls_per_column.sum()

# Calculate the total number of unique records that contain at least one null value
total_unique_records_with_nulls = len(df[df.isnull().any(axis=1)].drop_duplicates())

print("Nulls in each feature:")
print(nulls_per_column)
print("\nTotal nulls in the DataFrame:", total_nulls)
print("\nTotal records with at least one null value: ", total_unique_records_with_nulls)

In [ ]:
df['Zoning'].unique()

In [ ]:
zoning_mapping = {
    "Agricole": 0,
    "Industriel": 1,
    "Immeuble": 2,
    "Maison": 3,
    "Villa": 4,
    "Service public": 5
}

df['Zoning_enc'] = df['Zoning'].map(zoning_mapping)


In [ ]:
# Compter pour chaque Ville + Type
counts = df.groupby(['Ville', 'Zoning_enc']).size().reset_index(name='count')

# Pour chaque Ville, garder le Type avec le max count
idx = counts.groupby('Ville')['count'].idxmax()
most_frequent_type_per_ville = counts.loc[idx][['Ville', 'Zoning_enc']].set_index('Ville')['Zoning_enc']

df['Zoning_enc'] = df['Zoning_enc'].fillna(df['Ville'].map(most_frequent_type_per_ville))

In [ ]:
df['Zoning_enc'].unique()

In [ ]:
df['Loti'] = df['Loti'].map({'oui':1, 'Non':0})
df['Titre'] = df['Titre'].map({'oui':1, 'Non':0})
df.head()

In [ ]:
df_sorted = df.sort_values(by='Prix', ascending=True)
df_sorted = df_sorted[['Prix', 'Lien']]
# Afficher les 10 premières lignes triées par prix
print(df_sorted.head(10))

In [ ]:
# Function to convert a single price to Dirhams
def convert_to_dirhams(price):
    if isinstance(price, str):
        # Replace commas with periods and non-breaking spaces with nothing
        price = price.replace(',', '.').replace('\xa0', '')
        # Remove ' DH' and convert to float
        return float(price.replace('DH', '').replace(' ', ''))
    else:
        # If not a string, return as is
        return price

# Apply the function to each value in the 'Prix' column
df['Prix'] = df['Prix'].apply(convert_to_dirhams)

# Convert the non-null values in the 'Prix' column to integer
df.loc[df['Prix'].notna(), 'Prix'] = df.loc[df['Prix'].notna(), 'Prix'].astype(int)

df.head()

In [ ]:
df = df[df['Prix']>10000]
df.shape[0]

In [ ]:
df.info()

In [ ]:
from rapidfuzz import fuzz, process

# Exemple : liste unique de quartiers
quartiers = df['Quartier'].unique()

# Dictionnaire de correspondance
corrections = {}

# Boucle pour comparer chaque quartier aux autres
for i, q1 in enumerate(quartiers):
    for q2 in quartiers[i+1:]:
        score = fuzz.ratio(q1.lower(), q2.lower())
        if score >= 85:  # seuil de similarité
            print(f"{q1} <--> {q2} : {score}% similaire")
            # On choisit la forme la plus courte comme forme standard
            standard = min(q1, q2, key=len)
            corrections[q1] = standard
            corrections[q2] = standard

print("\nDictionnaire de remplacement :")
print(corrections)


In [ ]:
corrections = {'Hay salam': 'Hay salam', 'Hay Salam': 'Hay salam', 'Jawhara': 'Jawhar',  'Bir Rami Ouest': 'Bir Rami', 'Bir Rami Est': 'Bir Rami', 'Massira 3': 'Massira', 'Massira 2': 'Massira', 'Centre Ville': 'Centre ville'}
# Fonction de remplacement
def corriger_quartier(q):
    return corrections.get(q, q)  # si pas dans le dict, garder original

df['Quartier'] = df['Quartier'].apply(corriger_quartier)


In [ ]:
# Calculer la moyenne du prix par quartier
quartier_mean = df.groupby('Quartier')['Prix'].mean()

# Associer chaque ligne à la moyenne de son quartier
df['Quartier_enc'] = df['Quartier'].map(quartier_mean)

In [ ]:
df.head()

<h2 style="color:black"> Using IQR </h2>

In [ ]:
Q1 = df["Prix"].quantile(0.25)
Q3 = df["Prix"].quantile(0.75)
IQR = Q3 - Q1
# Define a condition for an outlier
outlier_condition = (df["Prix"] < (Q1 - 1.5 * IQR)) | (df["Prix"] > (Q3 + 1.5 * IQR))
# Get the outliers
outliersQP = df[outlier_condition]

# Print the number of outliers
print("\nNumber of outliers in Prix:", outliersQP.shape[0])

In [ ]:
df["Surface_Totale"] = df["Surface_Totale"].astype(float)
Q1 = df["Surface_Totale"].quantile(0.25)
Q3 = df["Surface_Totale"].quantile(0.75)
IQR = Q3 - Q1
# Define a condition for an outlier
outlier_condition1 = (df["Surface_Totale"] < (Q1 - 1.5 * IQR)) | (df["Surface_Totale"] > (Q3 + 1.5 * IQR))
# Get the outliers
outliersQS = df[outlier_condition1]
# Print the number of outliers
print("\nNumber of outliers in Surface:", outliersQS.shape[0])

In [ ]:
plt.figure(figsize=(4, 10))

# Subplot 1: Surface
plt.subplot(1, 2, 1)
sns.boxplot(y=df["Surface_Totale"], color='green')
plt.title('IQR of Surface')
plt.ylim(0, 600000)  # Set the y-axis limits from 0 to 1000

# Subplot 2:'Prix'
plt.subplot(1, 2, 2)
sns.boxplot(y=df["Prix"],width=0.6, color='orange')
plt.title('IQR of Prix')
plt.ylim(0, 9000000)  # Set the y-axis limits from 0 to 1000

plt.tight_layout()
plt.show()

<h2 style="color:black"> Using z-score </h2>

In [ ]:
z_scores = np.abs(stats.zscore(df['Prix']))
threshold = 3
outliersZ = z_scores > threshold

# Print the outliers
print(df['Prix'][outliersZ].count())

In [ ]:
z_scores = np.abs(stats.zscore(df['Surface_Totale']))
threshold = 3
outliersZ = z_scores > threshold

# Print the outliers number
print(df['Prix'][outliersZ].count())

In [ ]:
outliersQP.describe()

In [ ]:
df = df[~outlier_condition]

In [ ]:
df.shape[0]

In [ ]:
Q1 = df["Surface_Totale"].quantile(0.25)
Q3 = df["Surface_Totale"].quantile(0.75)
IQR = Q3 - Q1
# Define a condition for an outlier
outlier_condition = (df["Surface_Totale"] < (Q1 - 1.5 * IQR)) | (df["Surface_Totale"] > (Q3 + 1.5 * IQR))
# Get the outliers
outliersQS = df[outlier_condition]
# Print the number of outliers
print("\nNumber of outliers in Surface:", outliersQS.shape[0])

In [ ]:
outliersQS.describe()

In [ ]:
df = df[~outlier_condition]

In [ ]:
df.shape[0]

In [ ]:
plt.figure(figsize=(6.6, 4.6))
plt.scatter( df['Surface_Totale'] ,df['Prix'] ,color='#0080ff', label='Training Data')
plt.xlabel("Surface_Totale")
plt.ylabel('Prix')
plt.xlim(0, 20000)
plt.ylim(0, 6000000)
plt.show()

In [ ]:
import numpy as np

# Remplacer les vrais NaN (np.nan) ou chaînes "nan"/"NaN" par None
df = df.replace(["nan", "NaN", np.nan], None)

# Liste des colonnes sans 'id'
columns = [col for col in df.columns if col != "id"]

# Construction de la requête
cols_str = ", ".join(f"`{col}`" for col in columns)
placeholders = ", ".join(["%s"] * len(columns))
insert_query = f"INSERT INTO terrain_Transformed ({cols_str}) VALUES ({placeholders})"

# Insertion dans MySQL
for _, row in df[columns].iterrows():
    cursor.execute(insert_query, tuple(row))

conn.commit()


In [ ]:
cursor.close()
conn.close()
